# Stage 5A — BERT inference на eval (GPU, Colab)

**Среда:** Google Colab, A100 / T4 (Runtime → GPU)  
**Вход:** `EVAL_BASELINE_PATH`, `RUBERT_BEST_CHECKPOINT_DIR`, `RUBERT_CALIBRATION_PATH` (`utils/config.py`)  
**Выход:** `BERT_EVAL_PREDS_PATH` → `predictions/bert_eval_preds.parquet`  

После выполнения скачайте `bert_eval_preds.parquet` и положите в `predictions/` локального проекта,  
затем запустите **`stage5b_agent_loop.ipynb`** (агент + метрики, без GPU).

Логика: `utils/stage5_eval.run_stage5_bert_inference()` + `utils/predict.predict_bert()` — **без langchain / langgraph**.

---
⚠️ **`eval_baseline.parquet` открывается первый раз — не запускайте этот ноутбук повторно без причины.**

## 0. Проверка GPU

## 1. Colab / Drive и `utils.config`

На Colab корень на Drive (как в `stage2_bert_finetune.ipynb`). `config_local` задаёт `BASE_DIR`, пути — из `utils/config.py`.

In [1]:
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


## 2. Установка зависимостей

In [3]:
import subprocess
import sys
from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/Colab_Notebooks/NLP_ODS_2026/yandex_relevance')
sys.path.insert(0, str(PROJECT_ROOT))


if IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')],
        check=True,
    )
else:
    subprocess.run(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            'torch',
            'transformers',
            'pandas',
            'pyarrow',
            'tqdm',
        ],
        check=True,
    )
print('Dependencies OK')

Dependencies OK


In [4]:
import importlib
import sys
import types
import warnings
import torch
import pandas as pd
import json
from pathlib import Path

warnings.filterwarnings('ignore')


from utils.calibration import load_calibration
from utils.stage5_eval import run_stage5_bert_inference

from utils.config import (
    BASE_DIR,
    BERT_EVAL_PREDS_PATH,
    EVAL_BASELINE_PATH,
    BERT_BEST_CHECKPOINT_DIR,
    BERT_CALIBRATION_PATH,
    STAGE3_COMPARISON_PATH,
    COL_BERT_MAX_PROBA,
    COL_BERT_PRED,
    TARGET,
    create_directories,
)

_REQUIRED_INPUTS = {
    'eval_baseline': EVAL_BASELINE_PATH,
    'calibration': BERT_CALIBRATION_PATH,
    'checkpoint': BERT_BEST_CHECKPOINT_DIR,
    'stage3_threshold': STAGE3_COMPARISON_PATH,
}
for label, p in _REQUIRED_INPUTS.items():
    path = Path(p)
    assert path.exists(), f'{label}: не найдено {path}'

create_directories()
calib = load_calibration(BERT_CALIBRATION_PATH)
print('PROJECT_ROOT:', BASE_DIR)
print(f'Temperature T = {calib["temperature"]:.4f}')
print('Output:', BERT_EVAL_PREDS_PATH)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PROJECT_ROOT: /content/drive/MyDrive/Colab_Notebooks/NLP_ODS_2026/yandex_relevance
Temperature T = 1.1091
Output: /content/drive/MyDrive/Colab_Notebooks/NLP_ODS_2026/yandex_relevance/predictions/bert_eval_preds.parquet
CUDA available: True
GPU: Tesla T4


## 3. BERT inference (`run_stage5_bert_inference`)

`utils.data_loader.make_org_text`, `utils.predict.predict_bert`, порог Stage 3. На A100 можно поднять `BATCH_SIZE` в `predict_bert` (по умолчанию 64).

In [5]:
BATCH_SIZE = 64

print('Device:', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

out_path = run_stage5_bert_inference(overwrite=True, batch_size=BATCH_SIZE)

eval_df = pd.read_parquet(out_path)

with open(STAGE3_COMPARISON_PATH, encoding="utf-8") as f:
    threshold = float(json.load(f)["confidence_threshold"])

acc = float((eval_df[COL_BERT_PRED] == eval_df[TARGET]).mean())
low_conf_n = int((eval_df[COL_BERT_MAX_PROBA] < threshold).sum())
n = len(eval_df)

print(f'Accuracy (BERT-only): {acc:.4f}')
print(f'Threshold={threshold}, low-confidence: {low_conf_n} / {n} ({100 * low_conf_n / n:.1f}%)')
print(f'Сохранено: {out_path}')
print(f'Колонки: {list(eval_df.columns)}')

Device: cuda


predict_bert: load model:   0%|          | 0/3 [00:00<?, ?phase/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

predict_bert: inference: 100%|██████████| 72/72 [06:25<00:00,  5.35s/batch]


Accuracy (BERT-only): 0.7714
Threshold=0.68, low-confidence: 970 / 4558 (21.3%)
Сохранено: /content/drive/MyDrive/Colab_Notebooks/NLP_ODS_2026/yandex_relevance/predictions/bert_eval_preds.parquet
Колонки: ['permalink', 'text', 'name', 'address', 'normalized_main_rubric_name_ru', 'label', 'relevance', 'reviews_summarized', 'prices_summarized', 'org_text', 'bert_pred', 'bert_proba1', 'bert_max_proba']
